## STEP 0 — Import libraries and load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Load your data
df = pd.read_csv('morphometric_data_39basins.csv')
print('Shape:', df.shape)
df.head()

## STEP 1 — Look at the correlation matrix
This shows WHY we need PCA — many parameters are highly correlated with each other.
Red = strong positive correlation, Blue = strong negative correlation.

In [ ]:
# Select only numeric columns (drop SubBasin)
X = df.drop('SubBasin', axis=1)

plt.figure(figsize=(14, 10))
corr = X.corr()
sns.heatmap(corr, annot=False, cmap='coolwarm', center=0, linewidths=0.3)
plt.title('Correlation Matrix of 22 Morphometric Parameters\n(Many are highly correlated — this is the problem PCA solves)', fontsize=13)
plt.tight_layout()
plt.show()
print('Notice how many dark red squares there are — those parameters carry the same information')

## STEP 2 — Standardize the data
Each parameter has different units. We make them all comparable by setting mean=0, std=1.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('BEFORE standardization:')
print(f'  Area mean: {X["Area_km2"].mean():.2f} km², std: {X["Area_km2"].std():.2f}')
print(f'  Relief mean: {X["Relief_m"].mean():.2f} m, std: {X["Relief_m"].std():.2f}')

X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)
print('\nAFTER standardization (everything is now in same scale):')
print(f'  Area mean: {X_scaled_df["Area_km2"].mean():.4f}, std: {X_scaled_df["Area_km2"].std():.4f}')
print(f'  Relief mean: {X_scaled_df["Relief_m"].mean():.4f}, std: {X_scaled_df["Relief_m"].std():.4f}')
print('\nNow every parameter is on equal footing!')

## STEP 3 — Run PCA
PCA finds the underlying patterns in the data.

In [ ]:
pca = PCA()
pca.fit(X_scaled)

# How much variance does each component explain?
explained = pca.explained_variance_ratio_ * 100
cumulative = np.cumsum(explained)

print('Variance explained by each Principal Component:')
print('-' * 50)
for i in range(6):
    bar = '█' * int(explained[i])
    print(f'PC{i+1}: {explained[i]:5.1f}%  cumulative: {cumulative[i]:5.1f}%  {bar}')

print(f'\nPC1 + PC2 alone explain {cumulative[1]:.1f}% of all variance!')
print('This means 2 components summarize 22 parameters very well')

## STEP 4 — Plot the explained variance (Scree Plot)
The elbow in the curve shows where adding more components stops helping much.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
ax1.bar(range(1, 8), explained[:7], color='steelblue', edgecolor='black')
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Variance Explained (%)')
ax1.set_title('Scree Plot\n(How much each PC explains)')
ax1.set_xticks(range(1, 8))
for i, v in enumerate(explained[:7]):
    ax1.text(i+1, v+0.5, f'{v:.1f}%', ha='center', fontsize=10)

# Cumulative plot
ax2.plot(range(1, 8), cumulative[:7], 'o-', color='darkred', linewidth=2, markersize=8)
ax2.axhline(y=70, color='green', linestyle='--', label='70% threshold')
ax2.axhline(y=80, color='orange', linestyle='--', label='80% threshold')
ax2.set_xlabel('Number of Components')
ax2.set_ylabel('Cumulative Variance Explained (%)')
ax2.set_title('Cumulative Variance\n(How many PCs do we need?)')
ax2.legend()
ax2.set_xticks(range(1, 8))

plt.tight_layout()
plt.show()
print('Your paper selected PC1 and PC2 because they together explain >70% of variance')

## STEP 5 — Component Loadings
This is the KEY output. It tells you which parameters belong to which group.
High loading = that parameter strongly represents that PC.

In [ ]:
loadings = pd.DataFrame(
    pca.components_[:3].T,
    index=X.columns,
    columns=['PC1', 'PC2', 'PC3']
).round(3)

print('COMPONENT LOADINGS TABLE:')
print('(This is Table 2 in your paper)')
print('-' * 50)
print(loadings.to_string())

print('\nPC1 GROUP — parameters with |loading| > 0.2 on PC1:')
pc1 = loadings[abs(loadings['PC1']) > 0.2]['PC1'].sort_values(key=abs, ascending=False)
print(pc1)

print('\nPC2 GROUP — parameters with |loading| > 0.2 on PC2:')
pc2 = loadings[abs(loadings['PC2']) > 0.2]['PC2'].sort_values(key=abs, ascending=False)
print(pc2)

## STEP 6 — Visualize the loadings
This plot shows clearly which parameters belong to PC1 (size group) and which to PC2 (steepness group).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# PC1 loadings
pc1_sorted = loadings['PC1'].sort_values()
colors1 = ['steelblue' if v > 0 else 'tomato' for v in pc1_sorted]
ax1.barh(pc1_sorted.index, pc1_sorted.values, color=colors1, edgecolor='black', linewidth=0.5)
ax1.axvline(x=0, color='black', linewidth=0.8)
ax1.set_title('PC1 Loadings\n(SIZE & NETWORK group)', fontsize=13, fontweight='bold')
ax1.set_xlabel('Loading value')
ax1.grid(axis='x', alpha=0.3)

# PC2 loadings  
pc2_sorted = loadings['PC2'].sort_values()
colors2 = ['steelblue' if v > 0 else 'tomato' for v in pc2_sorted]
ax2.barh(pc2_sorted.index, pc2_sorted.values, color=colors2, edgecolor='black', linewidth=0.5)
ax2.axvline(x=0, color='black', linewidth=0.8)
ax2.set_title('PC2 Loadings\n(STEEPNESS & SHAPE group)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Loading value')
ax2.grid(axis='x', alpha=0.3)

plt.suptitle('Which parameters load on PC1 vs PC2?', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('CONCLUSION:')
print('PC1 = dominated by size-related parameters (Area, Perimeter, Stream lengths)')
print('PC2 = dominated by steepness parameters (Relief, Slope, Ruggedness, Elongation)')
print('These two groups match exactly what your paper found!')

## STEP 7 — Biplot (the classic PCA visualization)
Shows where each sub-basin sits in PC1-PC2 space, and which parameters pull in which direction.

In [ ]:
pca_scores = pca.transform(X_scaled)

plt.figure(figsize=(12, 8))

# Plot sub-basins
plt.scatter(pca_scores[:, 0], pca_scores[:, 1], 
            c='steelblue', s=80, zorder=5, edgecolors='black', linewidths=0.5)

# Label each sub-basin
for i, name in enumerate(df['SubBasin']):
    plt.annotate(name, (pca_scores[i, 0], pca_scores[i, 1]), 
                fontsize=7, ha='center', va='bottom')

# Draw loading arrows for key parameters
scale = 3
key_params = ['Area_km2', 'Perimeter_km', 'Relief_m', 'Slope_deg', 
              'Ruggedness', 'ElongationRatio', 'DrainageDensity']
colors_arrow = {'Area_km2':'darkblue', 'Perimeter_km':'darkblue', 
                'Relief_m':'darkred', 'Slope_deg':'darkred',
                'Ruggedness':'darkred', 'ElongationRatio':'darkgreen',
                'DrainageDensity':'purple'}

for param in key_params:
    idx = list(X.columns).index(param)
    plt.arrow(0, 0, 
              pca.components_[0, idx]*scale, 
              pca.components_[1, idx]*scale,
              head_width=0.1, head_length=0.05,
              fc=colors_arrow[param], ec=colors_arrow[param], linewidth=1.5)
    plt.text(pca.components_[0, idx]*scale*1.15,
             pca.components_[1, idx]*scale*1.15,
             param, fontsize=9, color=colors_arrow[param], fontweight='bold')

plt.xlabel(f'PC1 ({explained[0]:.1f}% variance) — SIZE & NETWORK', fontsize=12)
plt.ylabel(f'PC2 ({explained[1]:.1f}% variance) — STEEPNESS & SHAPE', fontsize=12)
plt.title('PCA Biplot: Sub-basins in PC1-PC2 space\nArrows show which direction each parameter pulls', fontsize=13)
plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print('Blue arrows = SIZE parameters (PC1)')
print('Red arrows = STEEPNESS parameters (PC2)')
print('Sub-basins in upper right = large AND steep = highest flash flood risk!')